# Autonomous Payment Agent with Claude and DPX

This notebook shows how to build a Claude agent that autonomously executes a cross-border vendor payment — from oracle check to on-chain settlement — with no human step required.

**What we'll build:**
1. An agent that reasons about whether conditions are right to settle
2. Gets a binding fee quote
3. Screens the recipient through AML and sanctions checks
4. Executes the settlement
5. Returns a signed receipt

**No API key or wallet needed for sandbox mode.** All DPX pricing and compliance endpoints are free and require no authentication.

---

## Architecture

```
Claude (claude-sonnet-5)
  └── tool: check_oracle_conditions
  └── tool: get_fee_quote
  └── tool: screen_counterparty
  └── tool: execute_settlement
```

Each tool wraps a DPX REST endpoint. Claude decides the order of calls, handles error states, and reasons about whether to proceed at each step.

## Setup

In [ ]:
%pip install anthropic httpx python-dotenv --quiet

In [ ]:
import os
import json
import httpx
import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

SANDBOX = os.environ.get("SANDBOX", "true").lower() != "false"

# DPX base URLs — no auth required
ORACLE_URL      = "https://stability.untitledfinancial.com"
AGENT_URL       = "https://agent.untitledfinancial.com"
COMPLIANCE_URL  = "https://compliance.untitledfinancial.com"

print(f"Sandbox mode: {SANDBOX}")
print(f"Claude model: claude-sonnet-5")

## Define the DPX tools

Each tool maps to one DPX endpoint. We define them as Claude tool-use schemas, then implement the actual HTTP calls separately.

In [ ]:
tools = [
    {
        "name": "check_oracle_conditions",
        "description": (
            "Check whether global macro conditions are suitable for settlement. "
            "Returns STABLE, CAUTION, or UNSTABLE with a score (0-100) and AI reasoning. "
            "Always call this first — do not proceed if status is UNSTABLE."
        ),
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "get_fee_quote",
        "description": (
            "Get a binding fee quote for a settlement. The quote is valid for 300 seconds "
            "and returns a quoteId required for settlement execution."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "amount_usd": {
                    "type": "number",
                    "description": "Settlement amount in USD"
                },
                "has_fx": {
                    "type": "boolean",
                    "description": "True if this is a cross-currency payment (adds FX fee)"
                },
                "esg_score": {
                    "type": "number",
                    "description": "Counterparty ESG score 0-100 (optional, defaults to 75)"
                }
            },
            "required": ["amount_usd", "has_fx"]
        }
    },
    {
        "name": "screen_counterparty",
        "description": (
            "Screen a payment recipient for AML, sanctions (OFAC/EU/UN/UK), and FATF R16 compliance. "
            "Returns PROCEED, HOLD, or BLOCKED with a reason. "
            "Never proceed if decision is BLOCKED. Route to human review if HOLD."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "amount": {
                    "type": "number",
                    "description": "Payment amount in USD"
                },
                "recipient_address": {
                    "type": "string",
                    "description": "Recipient wallet address (0x...)"
                },
                "counterparty_name": {
                    "type": "string",
                    "description": "Legal name of the recipient entity"
                }
            },
            "required": ["amount", "recipient_address", "counterparty_name"]
        }
    },
    {
        "name": "execute_settlement",
        "description": (
            "Execute the settlement. Only call after oracle is STABLE/CAUTION, "
            "a valid quoteId has been obtained, and the counterparty has been screened with PROCEED. "
            "In sandbox mode, no real funds move."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "amount": {
                    "type": "number",
                    "description": "Settlement amount in USD"
                },
                "recipient_address": {
                    "type": "string",
                    "description": "Recipient wallet address"
                },
                "quote_id": {
                    "type": "string",
                    "description": "quoteId from get_fee_quote (valid 300s)"
                },
                "purpose": {
                    "type": "string",
                    "description": "Payment purpose (e.g. 'vendor-invoice', 'payroll')"
                }
            },
            "required": ["amount", "recipient_address", "quote_id", "purpose"]
        }
    }
]

print(f"Defined {len(tools)} tools for Claude")

## Implement the tool calls

Each function makes the actual HTTP request to DPX and returns the result.

In [ ]:
def check_oracle_conditions() -> dict:
    """GET /reliability — free, no auth."""
    resp = httpx.get(f"{ORACLE_URL}/reliability", timeout=10)
    resp.raise_for_status()
    return resp.json()


def get_fee_quote(amount_usd: float, has_fx: bool, esg_score: float = 75) -> dict:
    """GET /quote — binding 300s quote, free, no auth."""
    resp = httpx.get(
        f"{ORACLE_URL}/quote",
        params={"amountUsd": amount_usd, "hasFx": str(has_fx).lower(), "esgScore": esg_score},
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()


def screen_counterparty(amount: float, recipient_address: str, counterparty_name: str) -> dict:
    """POST /flow-check — AML + sanctions + FATF R16, free, no auth."""
    resp = httpx.post(
        f"{COMPLIANCE_URL}/flow-check",
        json={
            "amount": amount,
            "currency": "USD",
            "recipient": recipient_address,
            "name": counterparty_name,
        },
        timeout=15,
    )
    resp.raise_for_status()
    return resp.json()


def execute_settlement(
    amount: float,
    recipient_address: str,
    quote_id: str,
    purpose: str,
) -> dict:
    """POST /settle — sandbox mode by default."""
    resp = httpx.post(
        f"{AGENT_URL}/settle",
        json={
            "amount": amount,
            "sourceCurrency": "USD",
            "destinationCurrency": "USD",
            "recipientAddress": recipient_address,
            "purpose": purpose,
            "quoteId": quote_id,
            "sandbox": SANDBOX,
        },
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()


def dispatch_tool(tool_name: str, tool_input: dict) -> str:
    """Route a Claude tool call to the right function and return JSON."""
    if tool_name == "check_oracle_conditions":
        result = check_oracle_conditions()
    elif tool_name == "get_fee_quote":
        result = get_fee_quote(**tool_input)
    elif tool_name == "screen_counterparty":
        result = screen_counterparty(**tool_input)
    elif tool_name == "execute_settlement":
        result = execute_settlement(**tool_input)
    else:
        result = {"error": f"Unknown tool: {tool_name}"}
    return json.dumps(result)


print("Tool implementations ready")

## The agent loop

We send a payment task to Claude and let it decide which tools to call and in what order. Claude handles the reasoning at each step — whether the oracle is safe, whether the counterparty is clean, whether the quote is still valid.

In [ ]:
def run_payment_agent(task: str) -> str:
    """Run the payment agent on a task. Returns the final response text."""
    print(f"\nTask: {task}")
    print("=" * 60)

    messages = [{"role": "user", "content": task}]

    system = (
        "You are an autonomous payment agent. When asked to execute a payment, "
        "you must:\n"
        "1. Check oracle conditions first — abort if UNSTABLE.\n"
        "2. Get a binding fee quote.\n"
        "3. Screen the counterparty — abort if BLOCKED, escalate if HOLD.\n"
        "4. Execute the settlement using the quoteId from step 2.\n"
        "Always explain your reasoning at each step. "
        "If any step returns an error or blocking decision, explain clearly and stop."
    )

    while True:
        response = client.messages.create(
            model="claude-sonnet-5",
            max_tokens=4096,
            system=system,
            tools=tools,
            messages=messages,
        )

        # Collect any text Claude produced this turn
        for block in response.content:
            if block.type == "text" and block.text.strip():
                print(f"\nClaude: {block.text}")

        # Done — no tool calls
        if response.stop_reason == "end_turn":
            final = next(
                (b.text for b in response.content if b.type == "text"), ""
            )
            return final

        # Collect tool calls and dispatch them
        tool_calls = [b for b in response.content if b.type == "tool_use"]
        if not tool_calls:
            break

        tool_results = []
        for call in tool_calls:
            print(f"\n  → {call.name}({json.dumps(call.input)})")
            output = dispatch_tool(call.name, call.input)
            parsed = json.loads(output)
            print(f"  ← {json.dumps(parsed, indent=4)}")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": call.id,
                "content": output,
            })

        # Feed tool results back to Claude
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

    return ""


print("Agent loop ready")

## Run the agent

Give Claude a payment task in plain English. It will call the DPX tools in the correct order and return a complete settlement receipt.

In [ ]:
result = run_payment_agent(
    "Pay $25,000 USD to wallet address 0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045 "
    "for Acme GmbH (vendor invoice #INV-2026-0042). "
    "This is a same-currency USD payment."
)
print("\n" + "=" * 60)
print("Final result:")
print(result)

## What just happened

Claude called four DPX endpoints autonomously:

| Step | Tool | DPX endpoint | Auth |
|------|------|-------------|------|
| 1 | `check_oracle_conditions` | `GET /reliability` | None |
| 2 | `get_fee_quote` | `GET /quote` | None |
| 3 | `screen_counterparty` | `POST /flow-check` | None |
| 4 | `execute_settlement` | `POST /settle` | Sandbox |

Claude reasoned at each step:
- If the oracle returned `UNSTABLE`, it would have halted and explained why.
- If the compliance screen returned `BLOCKED`, it would have refused to proceed.
- If the quote expired mid-flow (>300s), it would have re-fetched.

This is a complete autonomous payment agent — no human in the loop, no hardcoded control flow.

## Try a payment that gets blocked

The compliance screen catches sanctioned counterparties. Pass a name that triggers a block to see how Claude handles it.

In [ ]:
result = run_payment_agent(
    "Pay $10,000 USD to wallet address 0x1111111111111111111111111111111111111111 "
    "for Blocked Entity Corp."
)
print("\nFinal result:", result)

## Going further

**Add ESG-aware routing** — fetch the counterparty's live ESG score before quoting and pass it to `get_fee_quote`. Higher scores mean lower fees; 100% of ESG fees fund on-chain impact programs.

**Add delegated spend limits** — use the DPX Policy Engine to let an orchestrator agent set budgets for sub-agents. See [Multi-agent payments →](https://docs.untitledfinancial.com/guides/multi-agent-payments).

**Use MCP instead of REST** — if you're running Claude Desktop or Cursor, install `@untitledfinancial/dpx-mcp` and call the same operations as tool calls with no HTTP wiring:

```json
{
  "mcpServers": {
    "dpx": { "command": "npx", "args": ["-y", "@untitledfinancial/dpx-mcp"] }
  }
}
```

**Go live** — set `SANDBOX=false` in `.env` and fund a Base mainnet wallet. The same agent code runs against real settlement with no changes.

---

Resources:
- [DPX Docs](https://docs.untitledfinancial.com)
- [For AI Builders](https://docs.untitledfinancial.com/guides/for-ai-builders)
- [MCP Tools Reference](https://docs.untitledfinancial.com/integrations/mcp)